In [41]:
import urllib.request
import time
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime

# **Data Collection**

In [42]:
# Base Retail Directory
BASE_DIR = "http://mlg.ucd.ie/modules/python/sources/retail/"
YEAR = 2025

# Number of pages per Q
PAGES_PER_QUARTER = {
    1: 16,
    2: 17,
    3: 17,
    4: 16
}

# Small delay between requests
DELAY = 0.1


## **Helper and Data Cleaning Functions**

In [44]:
def get_html(url):
    """
    Gets the HTML for a single page and returns the string.
    """
    response = urllib.request.urlopen(url)
    html = response.read().decode()
    return html

#=====================================================================
def clean_text(text):
    """
    Removes extra whitespace and non-breaking spaces.
    """
    return text.replace("\xa0", " ").strip()

#========================================================================
def parse_money(text):
    """
    Converts money strings to a float.

    Examples it can handle:
      '€ 7.50'  -> 7.50
      'EUR 35.00' -> 35.00
      '18.50' -> 18.50

    If it cannot convert, it returns None.
    """
    t = clean_text(text)
    t = t.replace("€", "")
    t = t.replace("EUR", "")
    t = t.replace(",", "")
    t = t.strip()

    return float(t)

#===========================================================================
def parse_quantity(text):
    """
    Converts quantities to integers.
    Assuming -1 signals a refund and not data entry error
    """
    t = clean_text(text)
    
    return int(t)

#=============================================================================
def parse_date(text):
    """
    Converts the date formats to YYYY-MM-DD format.

    Formats Supported
    2025-10-01  
    01 Oct 2025  
    01-10-2025   
    Oct 01, 2025 
    """
    t = clean_text(text)

    formats = ["%Y-%m-%d", "%d %b %Y", "%d-%m-%Y", "%b %d, %Y"]

    for fmt in formats:
        try:
            return datetime.strptime(t, fmt).date().isoformat()
        except:
            pass

    # If none of the formats worked
    return None

#============================================================================
def parse_customer_details(details_text):
    """
    Extracts customer_id, location, gender, and age_band from the "Customer Details" field.

    The labels are inconsistent on the site, so we check common variants like:
      'Customer ID: 2487' or 'Cust ID: 10219' or 'ID: 15687'
      'Location Galway' or 'City: Cork' or 'From: Offaly'
      'Age Range: 55-64' or 'Age Group: 35-44' or 'Age: 45-54'
    """
    result = {
        "customer_id": None,
        "location": None,
        "gender": None,
        "age_band": None
    }
    # Handle both <br> tags and \n
    details_text = details_text.replace("<br>", "\n").replace("<br/>", "\n").replace("<br />", "\n")
       
    # split into lines (BeautifulSoup gives "\n" where <br> existed)
    lines = details_text.split("\n")

    for line in lines:
        line = clean_text(line)

        # ----- Customer ID -----
        # handle "Customer ID: 2487" / "Cust ID: 10219" / "ID: 15687" / "ID 18141"
        if "ID" in line:
            # make it easier by removing known labels
            tmp = line.replace("Customer ID:", "").replace("Cust ID:", "").replace("ID:", "").replace("ID", "").strip()
            # now tmp might be just a number
            try:
                result["customer_id"] = int(tmp)
            except:
                pass

        # ----- Location -----
        if line.startswith("Location"):
            result["location"] = clean_text(line.replace("Location", "").replace(":", ""))
        if line.startswith("City"):
            result["location"] = clean_text(line.replace("City", "").replace(":", ""))
        if line.startswith("From"):
            result["location"] = clean_text(line.replace("From", "").replace(":", ""))

        # ----- Gender -----
        if line.startswith("Gender"):
            result["gender"] = clean_text(line.replace("Gender", "").replace(":", ""))

        # ----- Age band -----
        if line.startswith("Age Range"):
            result["age_band"] = clean_text(line.replace("Age Range", "").replace(":", ""))
        if line.startswith("Age Group"):
            result["age_band"] = clean_text(line.replace("Age Group", "").replace(":", ""))
        if line.startswith("Age Category"):
            result["age_band"] = clean_text(line.replace("Age Category", "").replace(":", ""))
        if line.startswith("Age"):
            result["age_band"] = clean_text(line.replace("Age", "").replace(":", ""))

    return result


## **Parse Table Function**


In [45]:
def parse_transaction_info(table):
    """
    Each transaction is stored as a small table with rows like:
      Product, Sale Date, Category, Quantity, Total Price, Total Profit, Payment Type, Customer Details

    This function converts that table into a Python dictionary (one row of data).
    """
    row = {
        "product": None,
        "sale_date": None,
        "category": None,
        "quantity": None,
        "total_price": None,
        "total_profit": None,
        "payment_type": None,
        "customer_id": None,
        "location": None,
        "gender": None,
        "age_band": None
    }

    # Each row is a <tr> with two <td> cells: label and value
    for tr in table.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) < 2:
            continue

        label = clean_text(tds[0].get_text())
        value = tds[1].get_text("\n")   # keep line breaks so customer details split nicely
        value = clean_text(value)

        # Match based on the label text
        if label.startswith("Product"):
            row["product"] = clean_text(tds[1].get_text(" "))
        elif label.startswith("Sale Date"):
            row["sale_date"] = parse_date(value)
        elif label.startswith("Category"):
            row["category"] = value
        elif label.startswith("Quantity"):
            row["quantity"] = parse_quantity(value)
        elif label.startswith("Total Price"):
            row["total_price"] = parse_money(value)
        elif label.startswith("Total Profit"):
            row["total_profit"] = parse_money(value)
        elif label.startswith("Payment Type"):
            row["payment_type"] = value
        elif label.startswith("Customer Details"):
            cust = parse_customer_details(value)
            row["customer_id"] = cust["customer_id"]
            row["location"] = cust["location"]
            row["gender"] = cust["gender"]
            row["age_band"] = cust["age_band"]

    return row


In [ ]:
def parse_page(html):
    """
    A page contains multiple sales.
    Each transaction is a <table class="transaction-info">.

    This finds all those tables and converts them into a list of dictionaries.
    """
    soup = BeautifulSoup(html, "html.parser")

    # Select every transaction table on the page
    tables = soup.select("table.transaction-info")

    rows = []
    for table in tables:
        rows.append(parse_transaction_info(table))

    return rows


## **Main Data Scraping Loop**

In [48]:
all_rows = []

for quarter, max_page in PAGES_PER_QUARTER.items():
    for page in range(1, max_page + 1):

        url = f"{BASE_DIR}{YEAR}-Q{quarter}-page{page:02d}.html"
        print("Scraping:", url)

        html = get_html(url)
        page_rows = parse_page(html)

        # add metadata columns so we know where each row came from
        for r in page_rows:
            r["year"] = YEAR
            r["quarter"] = quarter
            r["page"] = page
            r["source_url"] = url

        all_rows.extend(page_rows)
        time.sleep(DELAY)

retail_raw = pd.DataFrame(all_rows)
retail_raw.shape

Scraping: http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page01.html
Scraping: http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page02.html
Scraping: http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page03.html
Scraping: http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page04.html
Scraping: http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page05.html
Scraping: http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page06.html
Scraping: http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page07.html
Scraping: http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page08.html
Scraping: http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page09.html
Scraping: http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page10.html
Scraping: http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page11.html
Scraping: http://mlg.ucd.ie/modules/python/sources/retail/2025-Q1-page12.html
Scraping: http://mlg.ucd.ie/modules/python/sources/retail/2025-Q

(1900, 15)

## **Save Data to CSV file**

In [49]:
retail_raw.to_csv("retail_raw.csv", index=False, encoding="utf-8")
print("Saved: retail_raw_2025.csv")

Saved: retail_raw_2025.csv
